In [12]:
import numpy as np
import pandas as pd
np.random.seed(42)

# input features
cgpa = np.random.uniform(low=5.0, high=10.0, size=2500)
iq = np.random.normal(loc=100, scale=15, size=2500)
marks_12th = np.random.uniform(low=60, high=100, size=2500)
marks_10th = np.random.uniform(low=60, high=100, size=2500)

# generating probabilities
placement_prob = (cgpa/10 * 0.3 + iq/150 * 0.3 + marks_12th/100 * 0.2 + marks_10th/100 * 0.2) # the floating point values are the weightage/importance of each feature in the overall probability 
placed = np.random.binomial(1, placement_prob)

# Create DataFrame
student_data = pd.DataFrame({
    'CGPA': np.round(cgpa, 2),
    'IQ': np.round(iq),
    'Marks_12th': np.round(marks_12th, 2),
    'Marks_10th': np.round(marks_10th, 2),
    'Placed': placed
})

student_data.head()

,CGPA,IQ,Marks_12th,Marks_10th,Placed
0,6.87,108.0,89.67,81.37,0
1,9.75,110.0,87.01,92.95,1
2,8.66,108.0,68.24,64.13,0
3,7.99,101.0,74.10,97.72,1
4,5.78,97.0,65.04,91.80,1


In [13]:
# Shape of the dataset
student_data.shape

(2500, 5)

In [14]:
# Splitting the data
X = student_data.drop(columns=['Placed'])
y = student_data['Placed']

In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class CustomDataset(Dataset):
    def __init__(self: CustomDataset, features: np.ndarray, labels: np.ndarray) -> None:
        scaler = StandardScaler() # global pre-processings and feature engineering techniques
        self.features = scaler.fit_transform(X = features) # applies feature wise (axis = 0)
        self.labels = labels
        
        self.features = torch.tensor(self.features, device='cpu', requires_grad=False)
        self.labels = torch.tensor(self.labels, device='cpu', requires_grad=False)

    def __len__(self: CustomDataset) -> int:
        return self.features.shape[0]

    def __getitem__(self: CustomDataset, index):
        return self.features[index], self.labels[index] # the shape of these tensors gets changed based on the type of data you are working with (tabular → (batch_size, no_features), image → (batch_size, no_rows, no_cols), text → (batch_size, no_tokens))

In [16]:
dataset = CustomDataset(
    features=np.array(X),
    labels=np.array(y)
)

# length of dataset
len(dataset)

2500

In [17]:
# Accessing a row
index = int(input("Enter your index → "))
features, labels = dataset[index]

print(f"Features at index {index}:", features)
print(f"Labels at index {index}:", labels)

Features at index 5: tensor([-1.1790, -0.1103, -1.2561, -1.1101], dtype=torch.float64)
Labels at index 5: tensor(0)


In [18]:
from torch.utils.data import BatchSampler, RandomSampler
sampler = RandomSampler(data_source=dataset)
batch_sampler = BatchSampler(sampler=sampler, batch_size=5, drop_last=False)

# Creating an object of dataset class → split the data into train and test then create a seperate dataloaders for each
dataloader = DataLoader(
    dataset=dataset,
    batch_size=5,
    # shuffle=True, # false for data like `Time Series`
    sampler=sampler,
    batch_sampler=None,
    drop_last=False,
    pin_memory=True,
    num_workers=0 # if you mention more than zero workers then you will not be able to print the data
)

# Returning the generator
dataloader

##### sampler → 
A `sampler` defines the strategy to draw **individual samples** from the dataset. It yields one index at a time. It tells the DataLoader *which single item* to pick next <br>
`SequentialSampler`, `RandomSampler`, `WeightedRandomSampler` → Yields indices based on given probabilities (useful for imbalanced datasets).<br>
When you specify a `sampler` and a `batch_size`, the DataLoader will draw `batch_size` number of individual indices from the `sampler` and collate those individual items into a single batch.

##### batch_sampler →
A `batch_sampler` defines the strategy to draw a **whole batch of samples** at once. It yields a list (or batch) of indices at a time. It tells the DataLoader *which group of items* to pick next for a complete batch. <br>
<i>You use a `batch_sampler` when you need fine-grained control over which specific items end up in the same batch. For example, in natural language processing, you might want to group sentences of similar lengths into the same batch to minimize padding `custom function → collate_fn parameter`.</i><br>
If you specify a `batch_sampler`, you **cannot** specify `batch_size`, `shuffle`, `sampler`, or `drop_last`. The `batch_sampler` takes over the entire responsibility of batching and sampling.

In [19]:
for idx, (batch_features, batch_labels) in enumerate(dataloader): # At every iteration, dataloader return whatever you returned in __getitem__ function
    print(batch_features, batch_labels)
    if idx == 3:
        break
    print("_" * 50)

tensor([[ 1.6819, -0.3100,  0.3617,  1.4228],
        [ 1.0699,  0.8884,  1.6448, -0.8544],
        [-1.4335, -0.1769,  1.3108,  1.5771],
        [ 1.5031, -0.2434,  1.4643,  1.0358],
        [ 1.0974, -0.4432,  0.4684,  0.0484]], dtype=torch.float64) tensor([1, 1, 0, 1, 1])
__________________________________________________
tensor([[-1.1378, -1.3087,  0.1232,  1.5877],
        [-0.2437,  0.2226,  1.6942,  0.3340],
        [ 0.8223,  0.4889, -0.2663, -0.5873],
        [ 0.0245, -0.9758,  0.2377,  1.4572],
        [ 0.7948, -1.1089,  1.3958,  0.7404]], dtype=torch.float64) tensor([1, 0, 1, 1, 0])
__________________________________________________
tensor([[-0.6632,  1.2212, -0.2108, -0.2576],
        [-1.3235, -0.7095, -0.5014,  1.2597],
        [-0.1749, -1.2421, -1.2683, -1.6726],
        [-1.5160,  0.0895, -0.0399,  0.6946],
        [ 1.5031,  0.9549, -0.2967, -0.0997]], dtype=torch.float64) tensor([1, 0, 0, 1, 1])
__________________________________________________
tensor([[-1.4473,  